In [1]:
import pandas as pd
from ipyfilechooser import FileChooser
from IPython.display import display
import os

dpath = "//10.69.168.1/crnldata/forgetting/Carla/"
print("🔍 Sélectionnez votre fichier CSV/Excel à analyser:")

fc1 = FileChooser(dpath, filter_pattern='*.csv',
                  title="<b>Sélectionnez votre fichier à analyser</b>")
display(fc1)

def update_file(chooser):
    global fichier_choisi, dpath
    fichier_choisi = chooser.selected
    dpath = os.path.dirname(fichier_choisi)
    %store dpath
    print(f"✅ Fichier sélectionné: {os.path.basename(fichier_choisi)}")

fc1.register_callback(update_file)


🔍 Sélectionnez votre fichier CSV/Excel à analyser:


FileChooser(path='\\10.69.168.1\crnldata\forgetting\Carla', filename='', title='<b>Sélectionnez votre fichier …

In [2]:
# Charger le fichier
df = pd.read_csv(fichier_choisi, sep=',')

print("📊 Informations sur les colonnes:")
print("Noms des colonnes:", df.columns.tolist())
print("\nAperçu des données:")
print(df.head())

# Nettoyer les noms de colonnes (enlever les espaces)
df.columns = df.columns.str.strip()
print("\n✅ Colonnes nettoyées:", df.columns.tolist())

print(f"\nNombre total d'enregistrements: {len(df)}")

# 1. REGROUPEMENT DES LABELS CONSÉCUTIFS IDENTIQUES
def regrouper_labels_consecutifs(df):
    """Regroupe les labels identiques qui se suivent"""
    episodes = []
    
    if len(df) == 0:
        return pd.DataFrame()
    
    # Initialiser le premier épisode
    current_label = df.iloc[0]['label']
    start_time = df.iloc[0]['time']
    total_duration = df.iloc[0]['duration']
    
    for i in range(1, len(df)):
        if df.iloc[i]['label'] == current_label:
            # Même label, on accumule la durée
            total_duration += df.iloc[i]['duration']
        else:
            # Changement de label, on sauvegarde l'épisode
            episodes.append({
                'start_time': start_time,
                'end_time': df.iloc[i-1]['time'] + df.iloc[i-1]['duration'],
                'label': current_label,
                'duration': total_duration
            })
            
            # Nouveau épisode
            current_label = df.iloc[i]['label']
            start_time = df.iloc[i]['time']
            total_duration = df.iloc[i]['duration']
    
    # Ajouter le dernier épisode
    episodes.append({
        'start_time': start_time,
        'end_time': df.iloc[-1]['time'] + df.iloc[-1]['duration'],
        'label': current_label,
        'duration': total_duration
    })
    
    return pd.DataFrame(episodes)

# Créer le DataFrame des épisodes regroupés
df_episodes = regrouper_labels_consecutifs(df)

print("\n✅ ÉPISODES REGROUPÉS:")
print(df_episodes.head(10))
print(f"\nNombre total d'épisodes: {len(df_episodes)}")

# 2. DURÉE MOYENNE DES LABELS PAR HEURE
print("\n⏱️ DURÉE MOYENNE DES ÉPISODES:")
duree_moyenne_par_heure = {}

for label in df_episodes['label'].unique():
    episodes_label = df_episodes[df_episodes['label'] == label]
    duree_moyenne_secondes = episodes_label['duration'].mean()
    duree_moyenne_minutes = duree_moyenne_secondes / 60
    duree_moyenne_par_heure[label] = duree_moyenne_minutes
    print(f"  {label}: {duree_moyenne_minutes:.2f} minutes par épisode")

# 3. POURCENTAGE DE CHAQUE LABEL SUR 24H
print("\n📊 RÉPARTITION SUR LA DURÉE TOTALE:")
duree_totale = df['duration'].sum()
heures_totales = duree_totale / 3600

for label in df['label'].unique():
    duree_label = df[df['label'] == label]['duration'].sum()
    pourcentage = (duree_label / duree_totale) * 100
    heures = duree_label / 3600
    print(f"  {label}: {pourcentage:.1f}% ({heures:.2f}h)")

print(f"\nDurée totale d'enregistrement: {heures_totales:.2f}h")

# 4. NOMBRE D'ÉPISODES DE CHAQUE TYPE
print("\n🔢 NOMBRE D'ÉPISODES:")
nombre_episodes = df_episodes['label'].value_counts().sort_index()
for label, count in nombre_episodes.items():
    print(f"  {label}: {count} épisodes")

# 5. STATISTIQUES DÉTAILLÉES PAR LABEL
print("\n📈 STATISTIQUES DÉTAILLÉES PAR LABEL:")
for label in df_episodes['label'].unique():
    episodes_label = df_episodes[df_episodes['label'] == label]
    print(f"\n  {label}:")
    print(f"    - Nombre d'épisodes: {len(episodes_label)}")
    print(f"    - Durée moyenne: {episodes_label['duration'].mean()/60:.2f} min")
    print(f"    - Durée médiane: {episodes_label['duration'].median()/60:.2f} min")
    print(f"    - Durée min: {episodes_label['duration'].min()/60:.2f} min")
    print(f"    - Durée max: {episodes_label['duration'].max()/60:.2f} min")
    print(f"    - Durée totale: {episodes_label['duration'].sum()/3600:.2f}h")

# Sauvegarder les résultats
output_file = fichier_choisi.replace('.csv', '_episodes_regroupes.csv')
df_episodes.to_csv(output_file, index=False)
print(f"\n💾 Fichier sauvegardé: {output_file}")

📊 Informations sur les colonnes:
Noms des colonnes: ['label', 'duration', 'time']

Aperçu des données:
    label  duration  time
0  NREM_2         5     0
1  NREM_2         5     5
2  Wake_1         5    10
3  Wake_1         5    15
4  Wake_1         5    20

✅ Colonnes nettoyées: ['label', 'duration', 'time']

Nombre total d'enregistrements: 35379

✅ ÉPISODES REGROUPÉS:
   start_time  end_time   label  duration
0           0        10  NREM_2        10
1          10        35  Wake_1        25
2          35       120  NREM_2        85
3         120       545  Wake_1       425
4         545       615  NREM_2        70
5         615       770  Wake_1       155
6         770      1120  NREM_2       350
7        1120      1125  Wake_1         5
8        1125      1130  NREM_2         5
9        1130      1445  Wake_1       315

Nombre total d'épisodes: 1505

⏱️ DURÉE MOYENNE DES ÉPISODES:
  NREM_2: 2.44 minutes par épisode
  Wake_1: 2.60 minutes par épisode
  REM_3: 0.56 minutes par épiso

In [3]:
# STATISTIQUES NORMALISÉES SUR 24H et 1H

# Calculer la durée totale de l'enregistrement
duree_totale_secondes = df['duration'].sum()
duree_totale_heures = duree_totale_secondes / 3600

print("=" * 70)
print("📊 STATISTIQUES NORMALISÉES")
print("=" * 70)
print(f"\n⏱️ Durée totale de l'enregistrement: {duree_totale_heures:.2f}h")

# Facteur de normalisation pour 24h
facteur_24h = 24 / duree_totale_heures

print("\n" + "=" * 70)
print("📈 STATISTIQUES SUR 24H (extrapolées)")
print("=" * 70)

for label in sorted(df_episodes['label'].unique()):
    print(f"\n🔹 {label}:")
    
    # Données brutes
    episodes_label = df_episodes[df_episodes['label'] == label]
    nb_episodes_reel = len(episodes_label)
    duree_totale_label = episodes_label['duration'].sum()
    duree_moyenne_label = episodes_label['duration'].mean()
    
    # Pourcentage sur 24h (reste le même)
    pourcentage = (duree_totale_label / duree_totale_secondes) * 100
    
    # Extrapolation sur 24h
    nb_episodes_24h = nb_episodes_reel * facteur_24h
    duree_totale_24h = duree_totale_label * facteur_24h / 3600  # en heures
    
    print(f"   • Pourcentage sur 24h: {pourcentage:.1f}%")
    print(f"   • Durée totale sur 24h: {duree_totale_24h:.2f}h ({duree_totale_24h*60:.0f} min)")
    print(f"   • Nombre d'épisodes sur 24h: {nb_episodes_24h:.1f}")
    print(f"   • Durée moyenne par épisode: {duree_moyenne_label/60:.2f} min")

print("\n" + "=" * 70)
print("⏰ STATISTIQUES SUR 1H (extrapolées)")
print("=" * 70)

# Facteur de normalisation pour 1h
facteur_1h = 1 / duree_totale_heures

for label in sorted(df_episodes['label'].unique()):
    episodes_label = df_episodes[df_episodes['label'] == label]
    
    # Données brutes
    nb_episodes_reel = len(episodes_label)
    duree_totale_label = episodes_label['duration'].sum()
    
    # Extrapolation sur 1h
    nb_episodes_1h = nb_episodes_reel * facteur_1h
    duree_minutes_1h = (duree_totale_label * facteur_1h) / 60
    
    print(f"\n🔹 {label}:")
    print(f"   • Durée par heure: {duree_minutes_1h:.2f} min")
    print(f"   • Nombre d'épisodes par heure: {nb_episodes_1h:.2f}")

print("\n" + "=" * 70)
print("📋 RÉSUMÉ COMPARATIF")
print("=" * 70)

# Créer un tableau récapitulatif
resume_data = []
for label in sorted(df_episodes['label'].unique()):
    episodes_label = df_episodes[df_episodes['label'] == label]
    nb_episodes_reel = len(episodes_label)
    duree_totale_label = episodes_label['duration'].sum()
    duree_moyenne_label = episodes_label['duration'].mean()
    pourcentage = (duree_totale_label / duree_totale_secondes) * 100
    
    resume_data.append({
        'Label': label,
        'Pourcentage_24h': pourcentage,
        'Heures_24h': duree_totale_label * facteur_24h / 3600,
        'Episodes_24h': nb_episodes_reel * facteur_24h,
        'Minutes_1h': (duree_totale_label * facteur_1h) / 60,
        'Episodes_1h': nb_episodes_reel * facteur_1h,
        'Duree_moyenne_episode_min': duree_moyenne_label / 60
    })

df_resume = pd.DataFrame(resume_data)

# Afficher le tableau formaté
df_resume_display = df_resume.copy()
df_resume_display['Pourcentage_24h'] = df_resume_display['Pourcentage_24h'].apply(lambda x: f"{x:.1f}%")
df_resume_display['Heures_24h'] = df_resume_display['Heures_24h'].apply(lambda x: f"{x:.2f}h")
df_resume_display['Episodes_24h'] = df_resume_display['Episodes_24h'].apply(lambda x: f"{x:.1f}")
df_resume_display['Minutes_1h'] = df_resume_display['Minutes_1h'].apply(lambda x: f"{x:.2f}")
df_resume_display['Episodes_1h'] = df_resume_display['Episodes_1h'].apply(lambda x: f"{x:.2f}")
df_resume_display['Duree_moyenne_episode_min'] = df_resume_display['Duree_moyenne_episode_min'].apply(lambda x: f"{x:.2f}")

print("\n", df_resume_display.to_string(index=False))

# Sauvegarder le tableau récapitulatif
output_resume = fichier_choisi.replace('.csv', '_statistiques_resume.csv')
df_resume.to_csv(output_resume, index=False, sep=';')

print("\n" + "=" * 70)
print(f"💾 Fichiers sauvegardés:")
print(f"   • Épisodes regroupés: {output_file}")
print(f"   • Statistiques résumées: {output_resume}")
print("=" * 70)

📊 STATISTIQUES NORMALISÉES

⏱️ Durée totale de l'enregistrement: 49.14h

📈 STATISTIQUES SUR 24H (extrapolées)

🔹 NREM_2:
   • Pourcentage sur 24h: 46.9%
   • Durée totale sur 24h: 11.26h (676 min)
   • Nombre d'épisodes sur 24h: 276.4
   • Durée moyenne par épisode: 2.44 min

🔹 REM_3:
   • Pourcentage sur 24h: 8.2%
   • Durée totale sur 24h: 1.97h (118 min)
   • Nombre d'épisodes sur 24h: 210.0
   • Durée moyenne par épisode: 0.56 min

🔹 Wake_1:
   • Pourcentage sur 24h: 44.9%
   • Durée totale sur 24h: 10.77h (646 min)
   • Nombre d'épisodes sur 24h: 248.6
   • Durée moyenne par épisode: 2.60 min

⏰ STATISTIQUES SUR 1H (extrapolées)

🔹 NREM_2:
   • Durée par heure: 28.15 min
   • Nombre d'épisodes par heure: 11.52

🔹 REM_3:
   • Durée par heure: 4.92 min
   • Nombre d'épisodes par heure: 8.75

🔹 Wake_1:
   • Durée par heure: 26.93 min
   • Nombre d'épisodes par heure: 10.36

📋 RÉSUMÉ COMPARATIF

  Label Pourcentage_24h Heures_24h Episodes_24h Minutes_1h Episodes_1h Duree_moyenne_episo